# P1 lab: tokens, truncation, temperature and cost

Lesson: [`concepts/P1_llm_fundamentals.md`](../concepts/P1_llm_fundamentals.md), Part H.1. Concept card: [02 P1](../02_RAG_MASTER_LEARNING_PLAN.md#p1).

**How this lab works (blank-file first).**
- Each task has a **task cell** (what to build), an **empty code cell** for you, and a **CHECK cell** (`# CHECK`, do not edit). Run the CHECK cell after your code. If it fails, read the message and fix your code.
- This notebook contains **no solution code**. Write every line yourself. You may look up library docs (API names are listed in the lesson, H.1). The reference script in `sources/verification/P1/` is a solution: open it only after an honest attempt and a hint from the tutor (CR-04).
- Where a task says **Predict first**, write the number in the prediction cell **before** you run anything.
- **No API calls** in this lab (no Groq, no Cohere). Never print values from `.env`.
- Kernel: the project **`.venv` (Python 3.12)**, as in `labs/O2_kernel_check.ipynb`.

## Setup (given)
Run this cell. It names the input file and defines `_fp`, a fingerprint that the CHECK cells use so the expected answers stay hidden until you have predicted.

In [ ]:
import hashlib, json
PDF = r"C:\Yash\New-Work\Book RAG\upstream\RAG_Techniques\data\Understanding_Climate_Change.pdf"

def _fp(x):
    """Fingerprint of a value. The CHECK cells compare fingerprints, not visible answers."""
    return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()[:16]

print(PDF)

## Task 1: load the PDF text

Build two variables:
- `pages`: a `list[str]`, one string per PDF page, from PyMuPDF's `page.get_text()`;
- `full`: the pages joined with a single newline `"\n"` (the same convention as the acceptance suite).

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 1 (do not edit)
assert isinstance(pages, list) and len(pages) == 33, f"expected 33 pages, got {len(pages) if isinstance(pages, list) else type(pages)}"
assert isinstance(full, str) and len(full) == 72593, f"expected 72,593 characters, got {len(full)}. Did you join with exactly one newline?"
assert full.startswith(pages[0]), "full must start with page 1"
print("Task 1 OK:", len(pages), "pages,", len(full), "characters")

## Task 2: count tokens with two tiktoken encodings

**Predict first.** Page 1 has 2,424 characters. Using "characters ÷ 4", how many tokens do you expect? Put your number in the next cell **before** counting.

In [ ]:
my_prediction_page1_tokens = None   # replace None with your number, BEFORE you count

Now build:
- `windows3`: a list of **three** strings, the plain character slices `full[0:1000]`, `full[800:1800]`, `full[1600:2600]` (size 1,000, overlap 200);
- `counts_cl100k`: a dict with keys `"page1"`, `"w0"`, `"w1"`, `"w2"` and the number of `cl100k_base` tokens of `pages[0]` and of each window;
- `counts_o200k`: the same dict for `o200k_base`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 2 (do not edit)
assert my_prediction_page1_tokens is not None, "write your prediction first (it is part of the evidence)"
assert len(windows3) == 3 and all(len(w) == 1000 for w in windows3), "windows3 must hold three 1,000-character strings"
assert windows3[1] == full[800:1800], "window 1 must start at character 800"
assert set(counts_cl100k) == set(counts_o200k) == {"page1", "w0", "w1", "w2"}, "use the keys page1, w0, w1, w2"
assert _fp(counts_cl100k) == "635dd86284b4383e", f"cl100k counts are not right yet: {counts_cl100k}"
assert _fp(counts_o200k) == "64e7448dbaeab835", f"o200k counts are not right yet: {counts_o200k}"
err = (my_prediction_page1_tokens - counts_cl100k["page1"]) / counts_cl100k["page1"]
print(f"Task 2 OK. Your prediction {my_prediction_page1_tokens} vs {counts_cl100k['page1']} cl100k tokens: {err:+.0%}")

## Task 3: count the same texts with the embedder's own tokenizer

Load the embedding model `SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")` into a variable named `model` (it is in your local cache; no download).

Build `counts_wordpiece` (same keys as Task 2): the number of WordPiece **text** tokens, **without** `[CLS]` and `[SEP]`.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 3 (do not edit)
assert _fp(counts_wordpiece) == "e07e48d2b998befc", \
    f"WordPiece counts are not right yet: {counts_wordpiece}. Did you exclude [CLS] and [SEP]?"
for k in counts_wordpiece:
    print(f"{k:6s} cl100k={counts_cl100k[k]:4d}  o200k={counts_o200k[k]:4d}  WordPiece={counts_wordpiece[k]:4d}")
print("Task 3 OK")

**Explain the gap (evidence for the 02 card).** In 2–3 sentences: why do the three tokenizers give different counts for the same three windows?

*Template:* "The counts differ because ____. WordPiece gives fewer tokens here because ____. This matters for ____."

*Your answer:*

## Task 4: see the cut (MC-001)

Use this sentence (given):

In [ ]:
SENT = "Anthropogenic emissions are accelerating deglaciation in Greenland."

Build:
- `pieces_cl100k`: the list of text pieces (strings) that `cl100k_base` cuts `SENT` into, one per token;
- `pieces_wordpiece`: the list of WordPiece token strings for `SENT`, **including** `[CLS]` and `[SEP]`.

Print both and look at where the rare words are cut.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 4 (do not edit)
assert len(pieces_cl100k) == 12, f"expected 12 cl100k pieces, got {len(pieces_cl100k)}"
assert "".join(pieces_cl100k) == SENT, "the cl100k pieces must join back to the exact sentence (spaces included)"
assert len(pieces_wordpiece) == 16 and pieces_wordpiece[0] == "[CLS]" and pieces_wordpiece[-1] == "[SEP]", \
    f"expected 16 WordPiece pieces with [CLS] first and [SEP] last, got {len(pieces_wordpiece)}"
print("Task 4 OK")

**MC-001 in your own words.** One or two sentences, no notes.

*Template:* "The pieces above were cut by the ____. The embedding model only ____."

*Your answer:*

## Task 5: characters per token is not a constant

Write a function `chars_per_token(text, count_fn)` that returns `len(text) / count_fn(text)`.

Then write **your own** three short samples in a dict `my_samples` with keys `"code"` (a few lines of Python), `"numbers"` (a line full of numbers, dates and IDs) and `"non_english"` (one sentence in a language other than English).

Build `ratio_table`: a dict with the same keys, each value a tuple `(cl100k chars/token, WordPiece chars/token)`. Print it next to the whole-PDF cl100k ratio.

**Predict first** (write it in the markdown cell below): which of your samples will have the lowest characters per token?

*My prediction (before running):* the lowest chars/token will be ____ because ____.

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 5 (do not edit)
assert abs(chars_per_token("abcdefgh", lambda t: 2) - 4.0) < 1e-12, "chars_per_token must be len(text) / count_fn(text)"
assert set(my_samples) == {"code", "numbers", "non_english"}, "my_samples needs the keys code, numbers, non_english"
assert set(ratio_table) == set(my_samples) and all(len(v) == 2 for v in ratio_table.values())
doc_ratio = 72593 / 13026   # whole-PDF cl100k ratio, measured in the lesson (S3)
for k, (a, b) in ratio_table.items():
    print(f"{k:12s} cl100k {a:5.2f}   WordPiece {b:5.2f}   (whole PDF cl100k {doc_ratio:.2f})")
assert ratio_table["numbers"][0] < doc_ratio, "a numbers-heavy line should have fewer chars per token than English prose"
print("Task 5 OK")

## Task 6: silent truncation in the embedder

Build:
1. `max_len`: the model's maximum sequence length, read from `model` (do not type the number).
2. `long_text`: a passage from `full` that has **exactly 400** WordPiece text tokens (no `[CLS]`/`[SEP]`).
3. `prefix_text`: the start of `long_text` that has **exactly 254** WordPiece text tokens (254 + `[CLS]` + `[SEP]` = 256).
4. `n_ids_fed`: how many token IDs the model actually receives for `long_text` (ask the model, do not compute it by hand).
5. `cos_trunc`: the cosine similarity between the normalized embeddings of `long_text` and `prefix_text`.

Hint for 2–3: cut text at token boundaries using character offsets. If your prefix re-tokenizes to a different count, the cut fell inside a word: start the passage one word later and try again.

**Predict first:** what will `cos_trunc` be?

In [ ]:
my_prediction_cos_trunc = None   # replace None with your number, BEFORE you embed

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 6 (do not edit)
_count = lambda t: len(model.tokenizer(t, add_special_tokens=False)["input_ids"])
assert my_prediction_cos_trunc is not None, "write your prediction first"
assert max_len == 256, f"max_len should come from the model; got {max_len}"
assert _count(long_text) == 400, f"long_text has {_count(long_text)} WordPiece text tokens, not 400"
assert _count(prefix_text) == 254 and long_text.startswith(prefix_text), "prefix_text must be the first 254 tokens of long_text"
assert n_ids_fed == 256, f"the model should receive 256 IDs, got {n_ids_fed}"
assert _fp(f"{cos_trunc:.6f}") == "4b3345f916b5cb21", f"cos_trunc = {cos_trunc:.6f} is not the expected value yet; check both texts and normalization"
print(f"Task 6 OK. You predicted {my_prediction_cos_trunc}; measured {cos_trunc:.6f}")

**What does this mean for a chunk?** One sentence.

*Template:* "Text after token ____ of a chunk ____, so a fact placed there ____."

*Your answer:*

## Task 7: temperature (toy logits)

Write `softmax_T(z, T)` that returns the probabilities `exp(z / T)` normalised to sum to 1, for a NumPy array `z` and a temperature `T > 0`. It must not overflow for large logits.

**Predict first (CR-08):** with the logits below, p(YES) is 0.532 at T = 1. If T doubles to 2, does p(YES) go up or down, and to about what?

In [ ]:
import numpy as np
LABELS = ["YES", "NO", "PARTIAL", "MAYBE"]
Z = np.array([2.0, 1.5, 0.5, -1.0])   # toy logits, not from any model
my_prediction_pyes_T2 = None          # replace None with your number, BEFORE you run your function

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 7 (do not edit)
import numpy as np
assert my_prediction_pyes_T2 is not None, "write your prediction first"
p1 = softmax_T(Z, 1.0)
assert abs(p1.sum() - 1) < 1e-12
_r3 = lambda T: [round(float(x), 3) for x in softmax_T(Z, T)]
assert _fp(_r3(1.0)) == "7e1fd9d9de553076", f"T=1 gives {_r3(1.0)}; p(YES) should be 0.532"
assert _fp(_r3(0.5)) == "2721b58682ee8d6b" and _fp(_r3(2.0)) == "7ea799c6829cd9b5", "T=0.5 or T=2 is wrong: is it exp(z / T), not exp(z * T)?"
big = softmax_T(np.array([1000.0, 999.0]), 1.0)
assert np.all(np.isfinite(big)), "overflow: subtract the max logit before exp"
print(f"Task 7 OK. p(YES) at T=2 = {softmax_T(Z, 2.0)[0]:.3f}; you predicted {my_prediction_pyes_T2}")

## Task 8: price sheet and the cost of 100 queries

**Step 1, look it up (web).** Open Groq's current pricing and model pages for `openai/gpt-oss-20b`. Fill in the cell below with what you read **today**. Do not use a number from memory or from this course.

In [ ]:
PRICE_IN_PER_M = None    # US$ per 1M input tokens, from Groq's page
PRICE_OUT_PER_M = None   # US$ per 1M output tokens, from Groq's page
CONTEXT_WINDOW = None    # tokens, from Groq's model page
PRICE_SOURCE_URL = None  # the page you read
PRICE_CHECKED_ON = None  # "YYYY-MM-DD"

**Step 2, write the formula.** Write `estimate_cost(n_in, n_out, price_in_per_m, price_out_per_m, n_queries=1)` that returns the cost in US$ (prices are per **1M** tokens, input and output priced separately).

**Step 3, predict, then compute.** Per query, use the measured v0 prompt (`N_IN = 976` tokens, lesson D.5) and the v0 output cap (`N_OUT_MAX = 512`, reasoning included). Write your prediction for 100 queries first, then compute `cost_100` with your function and your looked-up prices.

In [ ]:
N_IN, N_OUT_MAX = 976, 512           # from the lesson (S9) and 06 §3.1
my_prediction_cost_100_usd = None    # replace None with your guess, BEFORE computing

In [ ]:
# Your code here (blank-file first: write it yourself; no copying from p1_measure.py).

In [ ]:
# CHECK Task 8 (do not edit)
import math, re
assert all(isinstance(v, (int, float)) and v > 0 for v in (PRICE_IN_PER_M, PRICE_OUT_PER_M)), "fill both prices"
assert isinstance(CONTEXT_WINDOW, int) and CONTEXT_WINDOW > N_IN + N_OUT_MAX, "fill the context window (an int)"
assert isinstance(PRICE_SOURCE_URL, str) and PRICE_SOURCE_URL.startswith("http"), "record the page you read"
assert isinstance(PRICE_CHECKED_ON, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", PRICE_CHECKED_ON), "date as YYYY-MM-DD"
assert math.isclose(estimate_cost(1_000_000, 0, 1.0, 2.0), 1.0)
assert math.isclose(estimate_cost(0, 1_000_000, 1.0, 2.0), 2.0)
assert math.isclose(estimate_cost(1000, 300, 0.5, 1.5, n_queries=100), (100_000 * 0.5 + 30_000 * 1.5) / 1e6)
assert my_prediction_cost_100_usd is not None, "write your prediction first"
assert math.isclose(cost_100, estimate_cost(N_IN, N_OUT_MAX, PRICE_IN_PER_M, PRICE_OUT_PER_M, n_queries=100))
print(f"Task 8 OK. Upper bound for 100 queries: ${cost_100:.4f} (prices checked {PRICE_CHECKED_ON}); "
      f"you predicted ${my_prediction_cost_100_usd}")
print("Remember: billed tokens are higher than your own count (template overhead), see lesson B.3.")

## Wrap-up: evidence for the log

Answer each in one line, no notes.

1. *Three windows under two tokenizers:* "w0/w1/w2 were ____ / ____ / ____ cl100k tokens and ____ / ____ / ____ WordPiece tokens."
2. *The gap, in one sentence:* "____."
3. *Cost:* "I predicted $____ and computed $____ for 100 queries; my prediction was off because ____."
4. *One thing that would make the real bill higher than `cost_100`:* "____."

> Tell the tutor when you are done. The tutor grades `coding_independence_0_3` (Tasks 2, 6, 7) and `debugging_0_3` (any failed CHECK you fixed yourself).